In [1]:
# English stop words
stop_words = set(
    ['i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', 'your',
     'yours', 'yourself', 'yourselves', 'he', 'him', 'his', 'himself', 'she', 'her',
     'hers', 'herself', 'it', 'its', 'itself', 'they', 'them', 'their', 'theirs',
     'themselves', 'what', 'which', 'who', 'whom', 'this', 'that', 'these', 'those',
     'am', 'is', 'are', 'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had',
     'having', 'do', 'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if',
     'or', 'because', 'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about',
     'against', 'between', 'into', 'through', 'during', 'before', 'after', 'above',
     'below', 'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under',
     'again', 'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why',
     'how', 'all', 'any', 'both', 'each', 'few', 'more', 'most', 'other', 'some',
     'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too', 'very',
     's', 't', 'can', 'will', 'just', 'don', 'should', 'now', 'd', 'll', 'm', 'o',
     're', 've', 'y', 'ain', 'aren', 'couldn', 'didn', 'doesn', 'hadn', 'hasn', 'haven',
     'isn', 'ma', 'mightn', 'mustn', 'needn', 'shan', 'shouldn', 'wasn', 'weren', 'won',
     'wouldn', 'b', 'c', 'e', 'f', 'g', 'h', 'j', 'k', 'l', 'n', 'p', 'q', 'u', 'v',
     'w', 'x', 'z', 'us'])

# Java language keywords
java_keywords = set(
    ['abstract', 'assert', 'boolean', 'break', 'byte', 'case',
     'catch', 'char', 'class', 'const', 'continue', 'default', 'do', 'double',
     'else', 'enum', 'extends', 'false', 'final', 'finally', 'float', 'for', 'goto',
     'if', 'implements', 'import', 'instanceof', 'int', 'interface', 'long',
     'native', 'new', 'null', 'package', 'private', 'protected', 'public', 'return',
     'short', 'static', 'strictfp', 'super', 'switch', 'synchronized', 'this',
     'throw', 'throws', 'transient', 'true', 'try', 'void', 'volatile', 'while'])

In [2]:
!pip install inflection nltk javalang pygments


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path
from collections import namedtuple

# Dataset root directory
_DATASET_ROOT = Path("D:/NLP/bug_localization")  # Thay bằng đường dẫn của bạn

Dataset = namedtuple('Dataset', ['name', 'src', 'bug_repo', 'repo_url', 'features'])

aspectj = Dataset(
    'aspectj',
    _DATASET_ROOT / 'org.aspectj-bug43351/',
    _DATASET_ROOT / 'AspectJ.txt',
    "https://github.com/eclipse-aspectj/aspectj",
    _DATASET_ROOT / 'features_aspectj.csv'
)

eclipse = Dataset(
    'eclipse',
    _DATASET_ROOT / 'eclipse.platform.ui-johna-402445/',
    _DATASET_ROOT / 'Eclipse_Platform_UI.txt',
    "https://github.com/eclipse-platform/eclipse.platform.ui",
    _DATASET_ROOT / 'features_eclipse.csv'
)

swt = Dataset(
    'swt',
    _DATASET_ROOT / 'eclipse.platform.swt-xulrunner-31/',
    _DATASET_ROOT / 'SWT.txt',
    "https://github.com/eclipse-platform/eclipse.platform.swt",
    _DATASET_ROOT / 'features_swt.csv'
)

tomcat = Dataset(
    'tomcat',
    _DATASET_ROOT / 'tomcat-7.0.51/',
    _DATASET_ROOT / 'Tomcat.txt',
    "https://github.com/apache/tomcat",
    _DATASET_ROOT / 'features_tomcat/'
)

jdt = Dataset(
    'jdt',
    _DATASET_ROOT / 'eclipse.jdt-3.3/',
    _DATASET_ROOT / 'JDT.txt',
    "https://github.com/eclipse-jdt/eclipse.jdt",
    _DATASET_ROOT / 'features_jdt.csv'
)

birt = Dataset(
    'birt',
    _DATASET_ROOT / 'birt-2.1.0/',
    _DATASET_ROOT / 'Birt.txt',
    "https://github.com/eclipse-birt/birt",
    _DATASET_ROOT / 'features_birt.csv'
)

DATASET = aspectj  # Chọn dự án bạn muốn dùng

if __name__ == '__main__':
    print(DATASET.name, DATASET.src, DATASET.bug_repo)

aspectj D:\NLP\bug_localization\org.aspectj-bug43351 D:\NLP\bug_localization\AspectJ.txt


In [4]:
import pickle
import re
import string
import numpy as np
import inflection
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
from collections import defaultdict, Counter
import glob
import javalang
import pygments
from nltk.stem.porter import PorterStemmer
import os
import torch
import torch.nn as nn
import csv
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler
# from util import *
from collections import OrderedDict
from pygments.lexers import JavaLexer
from pygments.token import Token
from datetime import datetime

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\ACER\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


DATA PREPROCESSING:

Với các file Bug Report:
- Parser: đọc dữ liệu và xử lí dư liệu mượt hơn
- extract stack traces 
- extract stack traces remove 
- extracting specific pos tags from bug reports raw text
- tokenize
- split camel case 
- split camelcase
- normailze
- remove stop word
- remove java keyword
- stem

Với các file sourcecode:
- Parser: đọc dữ liệu và xử lí dữ liệu tốt hơn
- pos tagging giúp extract stack pos tags from comments
- tokenize
- split camelcase
- normalize
- reomve stop words
- remove java key words
- stem


1. Tiến hành đọc dữ liệu bug report và source code để chuẩn bị preprocessing

In [5]:
class BugReport:
    """Class representing each bug report"""
    __slots__ = ['summary', 'description', 'fixed_files', 'report_time', 'pos_tagged_summary',
                  'pos_tagged_description','stack_traces','stack_traces_remove']

    def __init__(self, summary, description, fixed_files, report_time):
        self.summary = summary
        self.description = description
        self.fixed_files = fixed_files
        self.report_time = report_time
        self.pos_tagged_summary = None
        self.pos_tagged_description = None
        self.stack_traces = None
        self.stack_traces_remove = None


In [6]:
class SourceFile:
    """Class representing each source file"""
    __slots__ = ['all_content', 'comments', 'class_names', 'attributes', 'method_names', 'variables', 'file_name',
                 'pos_tagged_comments', 'exact_file_name', 'package_name']

    def __init__(self, all_content, comments, class_names, attributes, method_names, variables, file_name,
                 package_name):
        self.all_content = all_content
        self.comments = comments
        self.class_names = class_names
        self.attributes = attributes
        self.method_names = method_names
        self.variables = variables
        self.file_name = file_name
        self.exact_file_name = file_name[0]
        self.package_name = package_name
        self.pos_tagged_comments = None

In [7]:
class Parser:
    __slots__ = ['name', 'src', 'bug_repo']

    def __init__(self, pro):
        self.name = pro.name
        self.src = pro.src
        self.bug_repo = pro.bug_repo

    def report_parser(self):
        reader = csv.DictReader(open(self.bug_repo, "r"), delimiter="\t")
        bug_reports = OrderedDict()
        for line in reader:
            line["report_time"] = datetime.strptime(line["report_time"], "%Y-%m-%d %H:%M:%S")

            # tách theo khoảng trắng thay vì ".java"
            fixed_files = line["files"].strip().split() if "files" in line else []
            x = []
            for f in fixed_files:
                if f and f.endswith(".java"):
                    f_norm = os.path.normpath(f).replace("\\", "/")
                    # chỉ lấy phần sau "src/"
                    if "src/" in f_norm:
                        f_norm = f_norm.split("src/")[-1]
                    x.append(f_norm)
            line["files"] = x

            bug_reports[line["bug_id"]] = BugReport(
                line["summary"], line["description"], line["files"], line["report_time"]
            )
        return bug_reports

    def src_parser(self):
        """Parse source code directory of a program and collect its java files"""
        src_addresses = glob.glob(str(self.src) + '/**/*.java', recursive=True)
        java_lexer = JavaLexer()
        src_files = OrderedDict()

        for src_file in src_addresses:
            with open(src_file, encoding='latin-1') as file:
                src = file.read()

            comments = ''
            class_names, attributes, method_names, variables = [], [], [], []

            parse_tree = None
            try:
                parse_tree = javalang.parse.parse(src)
                for path, node in parse_tree.filter(javalang.tree.VariableDeclarator):
                    if isinstance(path[-2], javalang.tree.FieldDeclaration):
                        attributes.append(node.name)
                    elif isinstance(path[-2], javalang.tree.VariableDeclaration):
                        variables.append(node.name)
            except:
                pass

            ind = False
            if parse_tree:
                if parse_tree.imports:
                    last_imp_path = parse_tree.imports[-1].path
                    src = src[src.index(last_imp_path) + len(last_imp_path) + 1:]
                elif parse_tree.package:
                    package_name = parse_tree.package.name
                    src = src[src.index(package_name) + len(package_name) + 1:]
                else:
                    ind = True
            else:
                ind = True

            lexed_src = pygments.lex(src, java_lexer)
            for i, token in enumerate(lexed_src):
                if token[0] in Token.Comment:
                    if ind and i == 0 and token[0] is Token.Comment.Multiline:
                        src = src[src.index(token[1]) + len(token[1]):]
                        continue
                    comments += token[1]
                elif token[0] is Token.Name.Class:
                    class_names.append(token[1])
                elif token[0] is Token.Name.Function:
                    method_names.append(token[1])

            if parse_tree and parse_tree.package:
                package_name = parse_tree.package.name
            else:
                package_name = None

    
            if self.name in ['aspectj', 'tomcat', 'eclipse', 'swt']:
                rel_path = os.path.relpath(src_file, start=self.src).replace("\\", "/")
                if "src/" in rel_path:
                    rel_path = rel_path.split("src/")[-1]
                src_files[rel_path] = SourceFile(
                    src, comments, class_names, attributes, method_names, variables,
                    [os.path.basename(src_file).split('.')[0]], package_name
                )
            else:
                if package_name:
                    src_id = (package_name + '.' + os.path.basename(src_file))
                else:
                    src_id = os.path.basename(src_file)
                src_files[src_id] = SourceFile(
                    src, comments, class_names, attributes, method_names, variables,
                    [os.path.basename(src_file).split('.')[0]], package_name
                )

        return src_files


2. Data preprocessing

In [8]:
class ReportPreprocessing:
    """Class preprocess bug reports"""
    __slots__ = ['bug_reports']

    def __init__(self, bug_reports):
        self.bug_reports = bug_reports

    def extract_stack_traces(self):
        """Extract stack traces from bug reports"""
        pattern = re.compile(r' at (.*?)\((.*?)\)')
        signs = ['.java', 'Unknown Source', 'Native Method']
        for report in self.bug_reports.values():
            st_canid = re.findall(pattern, report.description)
            st = [x for x in st_canid if any(s in x[1] for s in signs)]
            report.stack_traces = st

    def extract_stack_traces_remove(self):
        pattern = re.compile(r' at (.*?)\((.*?)\)')
        signs = ['.java', 'Unknown Source', 'Native Method']
        for report in self.bug_reports.values():
            st_canid = re.findall(pattern, report.description)
            st = [x for x in st_canid if any(s in x[1] for s in signs)]
            at = []
            for x in st:
                if (x[1] == 'Unknown Source'):
                    temp = 'Unknown'
                    y = x[0]+ '(' + temp
                else:
                    y = x[0] + '(' + x[1] + ')'
                at.append(y)
            report.stack_traces_remove = at

    def pos_tagging(self):
        """Extracing specific pos tags from bug reports raw_text"""
        for report in self.bug_reports.values():
            # Tokenizing using word_tokeize for more accurate pos-tagging
            sum_tok = nltk.word_tokenize(report.summary)
            desc_tok = nltk.word_tokenize(report.description)
            sum_pos = nltk.pos_tag(sum_tok)
            desc_pos = nltk.pos_tag(desc_tok)
            report.pos_tagged_summary = [token for token, pos in sum_pos if 'NN' in pos or 'VB' in pos]
            report.pos_tagged_description = [token for token, pos in desc_pos if 'NN' in pos or 'VB' in pos]

    def tokenize(self):
        """Tokenize bug report intro tokens"""
        for report in self.bug_reports.values():
            report.summary = nltk.wordpunct_tokenize(report.summary)
            report.description = nltk.wordpunct_tokenize(report.description)

    def _split_camelcase(self, tokens):
        # copy tokens
        returning_tokens = tokens[:]
        for token in tokens:
            split_tokens = re.split(fr'[{string.punctuation}]+', token)
            # if token is split into some other tokens
            if len(split_tokens) > 1:
                returning_tokens.remove(token)
                # camel case detection for new tokens
                for st in split_tokens:
                    camel_split = inflection.underscore(st).split('_')
                    if len(camel_split) > 1:
                        returning_tokens.append(st)
                        returning_tokens = returning_tokens + camel_split
                    else:
                        returning_tokens.append(st)
            else:
                camel_split = inflection.underscore(token).split('_')
                if len(camel_split) > 1:
                    returning_tokens = returning_tokens + camel_split
        return returning_tokens

    def split_camelcase(self):
        """Split camelcase indentifiers"""
        for report in self.bug_reports.values():
            report.summary = self._split_camelcase(report.summary)
            report.description = self._split_camelcase(report.description)
            report.pos_tagged_summary = self._split_camelcase(report.pos_tagged_summary)
            report.pos_tagged_description = self._split_camelcase(report.pos_tagged_description)

    def normalize(self):
        """remove punctuation, numbers and lowecase conversion"""
        # build a translate table for punctuation and number removal
        punctnum_table = str.maketrans({c: None for c in string.punctuation + string.digits})

        for report in self.bug_reports.values():
            summary_punctnum_rem = [token.translate(punctnum_table) for token in report.summary]
            desc_punctnum_rem = [token.translate(punctnum_table) for token in report.description]
            pos_sum_punctnum_rem = [token.translate(punctnum_table) for token in report.pos_tagged_summary]
            pos_desc_punctnum_rem = [token.translate(punctnum_table) for token in report.pos_tagged_description]
            report.summary = [token.lower() for token in summary_punctnum_rem if token]
            report.description = [token.lower() for token in desc_punctnum_rem if token]
            report.pos_tagged_summary = [token.lower() for token in pos_sum_punctnum_rem if token]
            report.pos_tagged_description = [token.lower() for token in pos_desc_punctnum_rem if token]

    def remove_stopwords(self):
        """removing stop word from tokens"""
        for report in self.bug_reports.values():
            report.summary = [token for token in report.summary if token not in stop_words]
            report.description = [token for token in report.description if token not in stop_words]
            report.pos_tagged_summary = [token for token in report.pos_tagged_summary if token not in stop_words]
            report.pos_tagged_description = [token for token in report.pos_tagged_description if token not in stop_words]

    def remove_java_keywords(self):
        """removing java language keywords from tokens"""
        for report in self.bug_reports.values():
            report.summary = [token for token in report.summary if token not in java_keywords]
            report.description = [token for token in report.description if token not in java_keywords]
            report.pos_tagged_summary = [token for token in report.pos_tagged_summary if token not in java_keywords]
            report.pos_tagged_description = [token for token in report.pos_tagged_description if token not in java_keywords]

    def stem(self):
        # stemming tokens
        stemmer = PorterStemmer()
        for report in self.bug_reports.values():
            report.summary = dict(
                zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in report.summary], report.summary]))
            report.description = dict(
                zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in report.description], report.description]))
            report.pos_tagged_summary = dict(
                zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in report.pos_tagged_summary], report.pos_tagged_summary]))
            report.pos_tagged_description = dict(
                zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in report.pos_tagged_description], report.pos_tagged_description]))

    def preprocess(self):
        self.extract_stack_traces()
        self.extract_stack_traces_remove()
        self.pos_tagging()
        self.tokenize()
        self.split_camelcase()
        self.normalize()
        self.remove_stopwords()
        self.remove_java_keywords()
        self.stem()

In [9]:
class SrcPreprocessing:
    """class to preprocess source code"""
    __slots__ = ['src_files']

    def __init__(self, src_files):
        self.src_files = src_files

    def pos_tagging(self):
        """Extracing specific pos tags from comments"""
        for src in self.src_files.values():
            # tokenize using word_tokenize
            comments_tok = nltk.word_tokenize(src.comments)
            comments_pos = nltk.pos_tag(comments_tok)
            src.pos_tagged_comments = [token for token, pos in comments_pos if 'NN' in pos or 'VB' in pos]

    def tokenize(self):
        """tokenize source code to tokens"""
        for src in self.src_files.values():
            src.all_content = nltk.wordpunct_tokenize(src.all_content)
            src.comments = nltk.wordpunct_tokenize(src.comments)

    def _split_camelcase(self, tokens):
        # copy token
        returning_tokens = tokens[:]
        for token in tokens:
            split_tokens = re.split(fr'[{string.punctuation}]+', token)
            # if token is split into some other tokens
            if len(split_tokens) > 1:
                returning_tokens.remove(token)
                # camelcase defect for new tokens
                for st in split_tokens:
                    camel_split = inflection.underscore(st).split('_')
                    if len(camel_split) > 1:
                        returning_tokens.append(st)
                        returning_tokens = returning_tokens + camel_split
                    else:
                        returning_tokens.append(st)
            else:
                camel_split = inflection.underscore(token).split('_')
                if len(camel_split) > 1:
                    returning_tokens = returning_tokens + camel_split
        return returning_tokens

    def split_camelcase(self):
        # Split camelcase indenti
        for src in self.src_files.values():
            src.all_content = self._split_camelcase(src.all_content)
            src.comments = self._split_camelcase(src.comments)
            src.class_names = self._split_camelcase(src.class_names)
            src.attributes = self._split_camelcase(src.attributes)
            src.method_names = self._split_camelcase(src.method_names)
            src.variables = self._split_camelcase(src.variables)
            src.pos_tagged_comments = self._split_camelcase(src.pos_tagged_comments)

    def normalize(self):
        "remove punctuation, number and lowercase conversion"
        # build a translate table for punctuation and number
        punctnum_table = str.maketrans({c: None for c in string.punctuation + string.digits})
        for src in self.src_files.values():
            content_punctnum_rem = [token.translate(punctnum_table) for token in src.all_content]
            comments_punctnum_rem = [token.translate(punctnum_table) for token in src.comments]
            classnames_punctnum_rem = [token.translate(punctnum_table) for token in src.class_names]
            attributes_punctnum_rem = [token.translate(punctnum_table) for token in src.attributes]
            methodnames_punctnum_rem = [token.translate(punctnum_table) for token in src.method_names]
            variables_punctnum_rem = [token.translate(punctnum_table) for token in src.variables]
            filename_punctnum_rem = [token.translate(punctnum_table) for token in src.file_name]
            pos_comments_punctnum_rem = [token.translate(punctnum_table) for token in src.pos_tagged_comments]

            src.all_content = [token.lower() for token in content_punctnum_rem if token]
            src.comments = [token.lower() for token in comments_punctnum_rem if token]
            src.class_names = [token.lower() for token in classnames_punctnum_rem if token]
            src.attributes = [token.lower() for token in attributes_punctnum_rem if token]
            src.method_names = [token.lower() for token in methodnames_punctnum_rem if token]
            src.variables = [token.lower() for token in variables_punctnum_rem if token]
            src.file_name = [token.lower() for token in filename_punctnum_rem if token]
            src.pos_tagged_comments = [token.lower() for token in pos_comments_punctnum_rem if token]

    def remove_stopwords(self):
        for src in self.src_files.values():
            src.all_content = [token for token in src.all_content if token not in stop_words]
            src.comments = [token for token in src.comments if token not in stop_words]
            src.class_names = [token for token in src.class_names if token not in stop_words]
            src.attributes = [token for token in src.attributes if token not in stop_words]
            src.method_names = [token for token in src.method_names if token not in stop_words]
            src.variables = [token for token in src.variables if token not in stop_words]
            src.file_name = [token for token in src.file_name if token not in stop_words]
            src.pos_tagged_comments = [token for token in src.pos_tagged_comments if token not in stop_words]

    def remove_javakeywords(self):
        for src in self.src_files.values():
            src.all_content = [token for token in src.all_content if token not in java_keywords]
            src.comments = [token for token in src.comments if token not in java_keywords]
            src.class_names = [token for token in src.class_names if token not in java_keywords]
            src.attributes = [token for token in src.attributes if token not in java_keywords]
            src.method_names = [token for token in src.method_names if token not in java_keywords]
            src.variables = [token for token in src.variables if token not in java_keywords]
            src.file_name = [token for token in src.file_name if token not in java_keywords]
            src.pos_tagged_comments = [token for token in src.pos_tagged_comments if token not in java_keywords]

    def stem(self):
        # stemming tokens
        stemmer = PorterStemmer()
        for src in self.src_files.values():
            src.all_content = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.all_content], src.all_content]))
            src.comments = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.comments], src.comments]))
            src.class_names = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.class_names], src.class_names]))
            src.attributes = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.attributes], src.attributes]))
            src.method_names = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.method_names], src.method_names]))
            src.variables = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.variables], src.variables]))
            src.file_name = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.file_name], src.file_name]))
            src.pos_tagged_comments = dict(zip(['stemmed', 'unstemmed'], [[stemmer.stem(token) for token in src.pos_tagged_comments], src.pos_tagged_comments]))


    def preprocess(self):
        self.pos_tagging()
        self.tokenize()
        self.split_camelcase()
        self.normalize()
        self.remove_stopwords()
        self.remove_javakeywords()
        self.stem()

In [10]:
def load_and_preprocess(dataset):
    parser = Parser(dataset)
    source_files = SrcPreprocessing(parser.src_parser())
    source_files.preprocess()
    source_files_string = source_files.src_files
    print(source_files_string)

    bug_reports = ReportPreprocessing(parser.report_parser())
    bug_reports.preprocess()
    report_strings = bug_reports.bug_reports
    print(report_strings)

    return source_files_string, report_strings





In [11]:
source_files, bug_reports =  load_and_preprocess(aspectj)

OrderedDict({'main/java/org/aspectj/ajde/Ajde.java': <__main__.SourceFile object at 0x000001B54D70D5B0>, 'main/java/org/aspectj/ajde/EditorAdapter.java': <__main__.SourceFile object at 0x000001B54D70EFF0>, 'main/java/org/aspectj/ajde/EditorListener.java': <__main__.SourceFile object at 0x000001B54D70D850>, 'main/java/org/aspectj/ajde/IconRegistry.java': <__main__.SourceFile object at 0x000001B54D70DA80>, 'main/java/org/aspectj/ajde/IdeUIAdapter.java': <__main__.SourceFile object at 0x000001B54D70DCB0>, 'main/java/org/aspectj/ajde/IRuntimeProperties.java': <__main__.SourceFile object at 0x000001B54D70DB60>, 'main/java/org/aspectj/ajde/IUIBuildMessageHandler.java': <__main__.SourceFile object at 0x000001B54D70DBD0>, 'main/java/org/aspectj/ajde/internal/BuildConfigListener.java': <__main__.SourceFile object at 0x000001B54D70DD90>, 'main/java/org/aspectj/ajde/internal/BuildConfigManager.java': <__main__.SourceFile object at 0x000001B54D70D540>, 'main/java/org/aspectj/ajde/internal/LstBuild

In [12]:
print(len(source_files))
print(len(bug_reports))

6886
593


In [13]:
print(source_files.keys())

odict_keys(['main/java/org/aspectj/ajde/Ajde.java', 'main/java/org/aspectj/ajde/EditorAdapter.java', 'main/java/org/aspectj/ajde/EditorListener.java', 'main/java/org/aspectj/ajde/IconRegistry.java', 'main/java/org/aspectj/ajde/IdeUIAdapter.java', 'main/java/org/aspectj/ajde/IRuntimeProperties.java', 'main/java/org/aspectj/ajde/IUIBuildMessageHandler.java', 'main/java/org/aspectj/ajde/internal/BuildConfigListener.java', 'main/java/org/aspectj/ajde/internal/BuildConfigManager.java', 'main/java/org/aspectj/ajde/internal/LstBuildConfigFileParser.java', 'main/java/org/aspectj/ajde/internal/LstBuildConfigFileUpdater.java', 'main/java/org/aspectj/ajde/internal/LstBuildConfigManager.java', 'main/java/org/aspectj/ajde/internal/StructureUtilities.java', 'main/java/org/aspectj/ajde/ui/AbstractIcon.java', 'main/java/org/aspectj/ajde/ui/AbstractIconRegistry.java', 'main/java/org/aspectj/ajde/ui/BuildConfigEditor.java', 'main/java/org/aspectj/ajde/ui/BuildConfigModel.java', 'main/java/org/aspectj/aj

In [14]:

from collections import defaultdict

# Giả sử bạn đã chạy main() và có bug_reports, src_files
# Tải GloVe (chạy lần đầu, sau comment out)
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-100")  # Thay bằng Common Crawl nếu có

# Chuẩn bị dữ liệu bổ sung
bug_to_fixed = {bug_id: report.fixed_files for bug_id, report in bug_reports.items()}
bug_to_date = {bug_id: report.report_time for bug_id, report in bug_reports.items()}
sorted_bugs = sorted(bug_to_date.keys(), key=lambda x: bug_to_date[x])

# Giả lập file_to_fixes từ bug_to_fixed (thay bằng parse Git nếu có)
file_to_fixes = defaultdict(list)
for bug_id, files in bug_to_fixed.items():
    for file_path in files:
        file_to_fixes[file_path].append((bug_id, bug_to_date[bug_id]))

# Hàm tạo text từ BugReport và SourceFile (dựa trên paper, page 4)
def get_bug_text(report):
    return report.summary['stemmed'] + report.description['stemmed']

def get_source_text(src):
    return src.comments['stemmed'] + src.class_names['stemmed'] + src.attributes['stemmed'] + \
           src.method_names['stemmed'] + src.variables['stemmed']

def normalize_path(path: str) -> str:
    # Bỏ slash Windows, giữ lại từ "org/" trở đi
    path = os.path.normpath(path).replace("\\", "/")
    if "org/" in path:
        path = path.split("org/")[-1]
        path = "org/" + path
    return path


# Tạo pairs
pairs = []
labels = []
for bug_id in sorted_bugs:
    report = bug_reports[bug_id]
    r_text = get_bug_text(report)
    r_date = bug_to_date[bug_id]
    
    # chuẩn hóa fixed_files
    fixed_set = {normalize_path(f) for f in report.fixed_files}
    
    for file_path in source_files.keys():
        src = source_files[file_path]
        s_text = get_source_text(src)

        # chuẩn hóa source path
        norm_src = normalize_path(file_path)
        
        label = 1 if norm_src in fixed_set else 0
        pairs.append((bug_id, file_path, r_text, s_text, r_date))
        labels.append(label)

pairs = np.array(pairs, dtype=object)
labels = np.array(labels)

print(f"Tổng số pairs: {len(pairs)}, Số pairs positive: {np.sum(labels)}, Tỷ lệ positive: {np.sum(labels)/len(labels):.4f}")




Tổng số pairs: 4083398, Số pairs positive: 1732, Tỷ lệ positive: 0.0004


In [74]:
print(pairs[:5])
print(labels[:10])


[['11280' 'main/java/org/aspectj/ajde/Ajde.java'
  list(['bug', 'view', 'mgmt', 'switch', 'perspect', 'lose', 'view', 'maxim', 'state', 'open', 'cv', 'repositori', 'view', 'java', 'perspect', 'click', 'titl', 'bar', 'maxim', 'resourc', 'perspect', 'come', 'back', 'java', 'perspect', 'cv', 'repositori', 'view', 'longer', 'maxim', 'view', 'maxim', 'stay', 'maxim'])
  list(['singleton', 'use', 'initi', 'ajd', 'ui', 'well', 'properti', 'requir', 'run', 'compil', 'user', 'must', 'call', 'ajd', 'init', 'anyth', 'getter', 'method', 'variou', 'properti', 'set', 'initi', 'also', 'defin', 'factori', 'get', 'ajcompil', 'instanc', 'author', 'mik', 'kersten', 'author', 'andi', 'clement', 'build', 'constructur', 'singleton', 'sub', 'class', 'initi', 'ajd', 'ui', 'set', 'compil', 'init', 'run', 'otherwis', 'util', 'run', 'project', 'main', 'project', 'properti', 'vm', 'use', 'loader', 'popul', 'classpath', 'output', 'path', 'jar', 'error', 'log', 'errorhandl', 'thread', 'run', 'process', 'unabl', 'st

3. Với project AspectJ, ta tiến hành data featuring, giả sử hiện tại có N bug reports, M source files,  ta lấy ra MxN cặp (r,s) và mỗi cặp nãy ta đánh giá qua 5 features gồm:
- Lexical similarity
- Semantic similarity
- Similar bug reports
- Code change history
- Bug fixing frequency

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import gensim.downloader as api

# Load GloVe embeddings (100d cho nhẹ)
glove_model = api.load("glove-wiki-gigaword-100")

def avg_glove_embedding(tokens, model):
    vecs = [model[w] for w in tokens if w in model]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

def extract_features(bug_reports, src_files, pairs, file_to_fixes, bug_to_date):
    # 1. TF-IDF vectorizer for lexical similarity
    bug_texts = {
        bid: " ".join(report.summary['stemmed'] + report.description['stemmed'])
        for bid, report in bug_reports.items()
    }
    src_texts = {
        sid: " ".join(src.all_content['stemmed'] + src.comments['stemmed'])
        for sid, src in src_files.items()
    }

    vectorizer = TfidfVectorizer()
    all_texts = list(bug_texts.values()) + list(src_texts.values())
    vectorizer.fit(all_texts)

    bug_tfidf = {bid: vectorizer.transform([txt]) for bid, txt in bug_texts.items()}
    src_tfidf = {sid: vectorizer.transform([txt]) for sid, txt in src_texts.items()}

    features = []

    for row in pairs:
        bug_id, src_id = row[0], row[1]

        if bug_id not in bug_reports or src_id not in src_files:
            continue

        f = []

        # 1. Lexical similarity (TF-IDF cosine)
        sim = cosine_similarity(bug_tfidf[bug_id], src_tfidf[src_id])[0][0]
        f.append(sim)

        # 2. Semantic similarity (GloVe embedding cosine)
        bug_tokens = bug_reports[bug_id].summary['stemmed'] + bug_reports[bug_id].description['stemmed']
        src_tokens = src_files[src_id].class_names['stemmed'] + \
                     src_files[src_id].method_names['stemmed'] + \
                     src_files[src_id].variables['stemmed']
        bug_vec = avg_glove_embedding(bug_tokens, glove_model).reshape(1, -1)
        src_vec = avg_glove_embedding(src_tokens, glove_model).reshape(1, -1)
        sem_sim = cosine_similarity(bug_vec, src_vec)[0][0]
        f.append(sem_sim)

        # 3. Similar bug reports
        
        report_time = bug_to_date[bug_id]
        past_fixes = [b for b, t in file_to_fixes.get(src_id, []) if t < report_time]
        similar_bug_score = 1.0 if len(past_fixes) > 0 else 0.0
        f.append(similar_bug_score)

        # 4. Code change history
        change_count = len(file_to_fixes.get(src_id, []))
        f.append(change_count)

        # 5. Bug fixing frequency
        bug_fix_count = sum(1 for b, _ in file_to_fixes.get(src_id, []))
        f.append(bug_fix_count)

        features.append(f)

    return np.array(features)


In [71]:
bug_to_date = {
    bug_id: report.report_time
    for bug_id, report in bug_reports.items()
}
from collections import defaultdict

file_to_fixes = defaultdict(list)

for bug_id, report in bug_reports.items():
    if hasattr(report, "fixed_files"):  
        for file_path in report.fixed_files:
            file_to_fixes[file_path].append((bug_id, report.report_time))


In [73]:
features = extract_features(bug_reports, source_files, pairs, bug_to_date, file_to_fixes)
print("Feature vector shape:", features.shape)
print("Ví dụ feature cho pair 0:", features[0])


Feature vector shape: (4083398, 5)
Ví dụ feature cho pair 0: [0.09015592 0.835581   0.         0.         0.        ]


4.Train và xử lí các vấn đề phát sinh

a. scale lại dữ liệu theo min max scale ( đã thực hiện ở trên trong quá tình trích xuất features) sử dụng focal loss để đánh giá dữ liệu mất cân bằng tốt hơn.

b. Sử dụng mạng DNN :  để xác định được những phần tử phi tuyến và những mối quan hệ phức tạp giữa nhưng features.

c. Mạng DNN này sẽ được áp dụng bootstrapping: tăng tốc quá tình huấn luyện thông qua chia dữ liệu thành các mini batch với batch size điều chỉnh theo thực nghiệm để tối ưu.

- Mẫu âm: cặp (s,r) mà ở đó bug không liên quan và gần nhưu không xuất hiện trong source file.
- Mẫu dương: cặp (s,r) mà ở đó bug có liên quan và gần nhưu chắc chắn xuất hiện trong source file.


Do khi chia thành các minibatches, sẽ có những mini batch mà nó không có mẫu dương do sự mất cân bằng dữ liệu, boostraping sẽ cố gắng xử lí điều đó bằng:
- gán trọng số cao hơn cho mẫu dương khi tạo mini batch, đảm bảo mẫu dương được lấy nhiều lần (oversampling).
- kết hợp với focal loss( đánh trọng số cao cho mẫu dương) để giảm ảnh hưởng của mẫu âm dễ dự đoán, tập trung mẫu dương khó phân loại.
- Model sẽ học một lần 64 mẫu để tránh quá tải bộ nhớ, và trong 64 mẫu đó ta chọn ngẫu nhiên các mẫu theo tỉ lệ 1:3

d. Chiến lược train: chia train test là 8:2, trong tập train, chia tập train và validation test theo tỉ lệ 8:2 nhằm đánh giá quá trình huấn luyện mô hình. Sau đó mới cho mô hình test trên tập test, đánh giá bằng MRR và MAP

- Đánh giá thể hiện mô hình bằng MRR và MAP.
- MRR: mean reciporal rank để đánh giá mô hình tìm thấy mẫu dương sớm hay muộn, bằng trung bình 1/(thứ tự tìm thấy mẫu dương đó)
- MAP: tại mỗi vị trí mô hình xác định được mẫu đúng, lấy tỉ lệ số mẫu đúng đã tìm được lúc đó với số mẫu đã xét đến hiện tại, lấy trung bình

In [87]:

import torch.optim as optim
import gc
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split


scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(features.astype(np.float32))
y = np.array(labels)


bug_ids = np.array([p[0] for p in pairs])


unique_bugs = np.unique(bug_ids)
train_bugs, test_bugs = train_test_split(unique_bugs, test_size=0.2, random_state=42)
train_bugs, val_bugs = train_test_split(train_bugs, test_size=0.2, random_state=42)

def select_by_bugs(X, y, pairs, bug_set):
    bug_set = set(bug_set)
    idx = [i for i, p in enumerate(pairs) if p[0] in bug_set]
    return X[idx], y[idx], pairs[idx]

X_train, y_train, pairs_train = select_by_bugs(X_scaled, y, pairs, train_bugs)
X_val, y_val, pairs_val = select_by_bugs(X_scaled, y, pairs, val_bugs)
X_test, y_test, pairs_test = select_by_bugs(X_scaled, y, pairs, test_bugs)


X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
y_val_t   = torch.tensor(y_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)
y_test_t  = torch.tensor(y_test, dtype=torch.float32)


class DNN(nn.Module):
    def __init__(self, input_dim):
        super(DNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),   
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1),
            nn.Sigmoid()
)

        
    def forward(self, x):
        return self.net(x)

model = DNN(X_scaled.shape[1])


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        BCE_loss = nn.BCELoss(reduction='none')(inputs, targets)
        pt = torch.exp(-BCE_loss)
        focal_loss = self.alpha * (1-pt)**self.gamma * BCE_loss
        return focal_loss.mean()

criterion = FocalLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)


pos_idx = np.where(y_train == 1)[0]
neg_idx = np.where(y_train == 0)[0]

def get_balanced_batch(pos_idx, neg_idx, batch_size=64, neg_ratio=3):
    n_pos = max(1, batch_size // (neg_ratio + 1))
    pos_samples = np.random.choice(pos_idx, n_pos, replace=True)
    n_neg = n_pos * neg_ratio
    neg_samples = np.random.choice(neg_idx, n_neg, replace=True)
    idx = np.concatenate([pos_samples, neg_samples])
    np.random.shuffle(idx)
    return idx


def evaluate_ranking(model, X, y, pairs, k_values=[1, 5, 10], batch_size=1024):
    model.eval()
    scores = []

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = X[i:i+batch_size]
            batch_scores = model(X_batch).cpu().numpy().flatten()
            scores.extend(batch_scores)

    results = {}
    for pair, score, label in zip(pairs, scores, y):
        bug_id, src_id = pair[0], pair[1]
        if bug_id not in results:
            results[bug_id] = []
        results[bug_id].append((src_id, score, label))

    topk_correct = {k: 0 for k in k_values}
    rr_sum, ap_sum = 0, 0
    n_bugs = len(results)

    for bug_id, lst in results.items():
        lst = sorted(lst, key=lambda x: x[1], reverse=True)
        labels = [l for _, _, l in lst]

        for k in k_values:
            if 1 in labels[:k]:
                topk_correct[k] += 1

        if 1 in labels:
            first_hit = labels.index(1) + 1
            rr_sum += 1 / first_hit

        hits, precisions = 0, []
        for i, l in enumerate(labels, start=1):
            if l == 1:
                hits += 1
                precisions.append(hits / i)
        if hits > 0:
            ap_sum += np.mean(precisions)

    metrics = {f"Top@{k}": topk_correct[k]/n_bugs for k in k_values}
    metrics["MRR"] = rr_sum / n_bugs
    metrics["MAP"] = ap_sum / n_bugs
    return metrics


In [90]:

epochs = 30
batch_size = 1024
best_val_map = 0
patience, patience_counter = 3, 0
best_model_state = None

for epoch in range(epochs):
    model.train()
    batch_losses = []
    for _ in range(len(X_train)//batch_size):
        idx = get_balanced_batch(pos_idx, neg_idx, batch_size, neg_ratio=3)
        X_batch, y_batch = X_train_t[idx], y_train_t[idx].view(-1,1)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())


    val_metrics = evaluate_ranking(model, X_val_t, y_val, pairs_val)
    print(f"Epoch {epoch+1}, Train Loss: {np.mean(batch_losses):.4f}, "
          f"Val MAP: {val_metrics['MAP']:.4f}, Val MRR: {val_metrics['MRR']:.4f}")


    if val_metrics["MAP"] > best_val_map:
        best_val_map = val_metrics["MAP"]
        best_model_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered")
            break

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



Epoch 1, Train Loss: 0.0233, Val MAP: 0.0550, Val MRR: 0.0967
Epoch 2, Train Loss: 0.0233, Val MAP: 0.0552, Val MRR: 0.0970
Epoch 3, Train Loss: 0.0233, Val MAP: 0.0554, Val MRR: 0.0973
Epoch 4, Train Loss: 0.0233, Val MAP: 0.0553, Val MRR: 0.0970
Epoch 5, Train Loss: 0.0233, Val MAP: 0.0555, Val MRR: 0.0973
Epoch 6, Train Loss: 0.0233, Val MAP: 0.0545, Val MRR: 0.0922
Epoch 7, Train Loss: 0.0233, Val MAP: 0.0552, Val MRR: 0.0970
Epoch 8, Train Loss: 0.0233, Val MAP: 0.0548, Val MRR: 0.0923
Early stopping triggered


In [91]:


test_metrics = evaluate_ranking(model, X_test_t, y_test, pairs_test)
print("Final Test metrics:", test_metrics)


Final Test metrics: {'Top@1': 0.09243697478991597, 'Top@5': 0.16806722689075632, 'Top@10': 0.31092436974789917, 'MRR': 0.15735535795728756, 'MAP': 0.10773545587780353}


ý tưởng mô hình tiếp theo: là mô hình học tăng cường
- Mục tiêu: cho mô hình quyết định xem file này là có bug này hay không ( 0 hoặc 1) với mỗi cặp (s,r).


Cách học:
- 2 hành động tại mỗi trạng thái: 0(k lquan bug) và 1(lquan bug)
- Reward Phần thưởng nếu dự đoán đúng , +1 cho dự đoán 1, +0.5 cho dự đoán 0
- Reward Phần thưởng nếu dự đoán sai -1
- Dùng mô hình DQN